In [0]:
# Read all 11 CSV files (July + August 2026) into one DataFrame
trips_raw = (
    spark.read
    .option("header", "true")                              # first row holds column names
    .csv("/Volumes/workspace/default/citibike/*/*.csv")    # * matches every month folder and every file
)

In [0]:
# Show each column's name and data type
trips_raw.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)



In [0]:
# Count total rows (this is an action, so Spark actually reads the files now)
trips_raw.count()

10239373

In [0]:
from pyspark.sql import functions as F

trips = (
    trips_raw
    # Convert time strings into real timestamp type
    .withColumn("started_at", F.to_timestamp("started_at"))
    .withColumn("ended_at", F.to_timestamp("ended_at"))
    # Convert coordinate strings into numbers
    .withColumn("start_lat", F.col("start_lat").cast("double"))
    .withColumn("start_lng", F.col("start_lng").cast("double"))
    .withColumn("end_lat", F.col("end_lat").cast("double"))
    .withColumn("end_lng", F.col("end_lng").cast("double"))
    # New column: ride duration in minutes (end seconds - start seconds) / 60
    .withColumn(
        "duration_min",
        (F.unix_timestamp("ended_at") - F.unix_timestamp("started_at")) / 60
    )
    # New column: calendar date of the ride (used later to join with weather)
    .withColumn("ride_date", F.to_date("started_at"))
    # New column: hour of day the ride started, 0-23 (used later for groupBy)
    .withColumn("start_hour", F.hour("started_at"))
)

trips.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- ride_date: date (nullable = true)
 |-- start_hour: integer (nullable = true)



In [0]:
# Peek at a few rows to sanity-check the new columns
trips.select(
    "started_at", "ended_at", "duration_min", "ride_date", "start_hour", "member_casual"
).show(5, truncate=False)

+-----------------------+-----------------------+------------------+----------+----------+-------------+
|started_at             |ended_at               |duration_min      |ride_date |start_hour|member_casual|
+-----------------------+-----------------------+------------------+----------+----------+-------------+
|2026-07-04 15:52:29.356|2026-07-04 16:03:08.021|10.65             |2026-07-04|15        |member       |
|2026-07-09 00:31:21.213|2026-07-09 00:35:10.506|3.816666666666667 |2026-07-09|0         |member       |
|2026-07-02 22:51:30.134|2026-07-02 22:56:13.056|4.716666666666667 |2026-07-02|22        |member       |
|2026-07-01 15:50:55.534|2026-07-01 15:58:18.824|7.383333333333334 |2026-07-01|15        |member       |
|2026-07-08 06:13:41.033|2026-07-08 06:33:30.627|19.816666666666666|2026-07-08|6         |member       |
+-----------------------+-----------------------+------------------+----------+----------+-------------+
only showing top 5 rows


In [0]:
trips_clean = (
    trips
    # Filter 1: drop rides missing a start or end station (e.g. dockless e-bike drop-offs)
    .filter(F.col("start_station_id").isNotNull() & F.col("end_station_id").isNotNull())
    # Filter 2: keep only realistic durations, between 1 minute and 24 hours
    .filter((F.col("duration_min") >= 1) & (F.col("duration_min") <= 24 * 60))
    # Filter 3: keep only rides that started in July-August 2026
    .filter(F.col("ride_date").between("2026-07-01", "2026-08-31"))
)

In [0]:
# Count rows before and after cleaning (both counts are actions)
raw_count = trips.count()
clean_count = trips_clean.count()
removed = raw_count - clean_count

print(f"Rows before filtering: {raw_count:,}")
print(f"Rows after filtering:  {clean_count:,}")
print(f"Rows removed:          {removed:,} ({removed / raw_count:.2%})")

Rows before filtering: 10,239,373
Rows after filtering:  10,203,608
Rows removed:          35,765 (0.35%)


In [0]:
# Group rides by start hour and rider type, then summarize each group
hourly = (
    trips_clean
    .groupBy("start_hour", "member_casual")
    .agg(
        F.count("*").alias("num_rides"),                          # how many rides in this group
        F.round(F.avg("duration_min"), 2).alias("avg_duration_min")  # average ride length
    )
    .orderBy("start_hour", "member_casual")                      # sort for readability
)

display(hourly)

start_hour,member_casual,num_rides,avg_duration_min
0,casual,48666,18.37
0,member,121319,12.9
1,casual,32551,18.1
1,member,74057,12.72
2,casual,21437,18.73
2,member,45824,12.73
3,casual,13924,18.49
3,member,29837,12.71
4,casual,11161,16.57
4,member,28195,11.69


Databricks visualization. Run in Databricks to view.

In [0]:
# Reshape: one row per hour, one column per rider type (a "wide" table)
hourly_wide = (
    trips_clean
    .groupBy("start_hour")
    .pivot("member_casual", ["member", "casual"])   # turn each rider type into its own column
    .count()                                        # fill each cell with the number of rides
    .orderBy("start_hour")
)

display(hourly_wide)

start_hour,member,casual
0,121319,48666
1,74057,32551
2,45824,21437
3,29837,13924
4,28195,11161
5,62918,12912
6,170614,26119
7,316201,44882
8,528445,71668
9,452706,80866


Databricks visualization. Run in Databricks to view.

In [0]:
# Departures: how many rides START at each station
departures = (
    trips_clean
    .groupBy(F.col("start_station_id").alias("station_id"))
    .agg(
        F.first("start_station_name", ignorenulls=True).alias("station_name"),
        F.count("*").alias("departures"),
    )
)

# Arrivals: how many rides END at each station
arrivals = (
    trips_clean
    .groupBy(F.col("end_station_id").alias("station_id"))
    .agg(F.count("*").alias("arrivals"))
)

# Join the two small aggregated tables on station_id.
# broadcast() ships the small table to every executor, so no shuffle is needed for the join.
station_flow = (
    departures
    .join(F.broadcast(arrivals), on="station_id", how="outer")
    .fillna(0, subset=["departures", "arrivals"])
    .withColumn("net_flow", F.col("arrivals") - F.col("departures"))  # >0 fills up, <0 drains
)

# Stations that lose the most bikes (most likely to run empty)
display(station_flow.orderBy("net_flow").limit(10))

station_id,station_name,departures,arrivals,net_flow
5343.10,Allen St & Hester St,14737,6038,-8699
5980.10,E 17 St & Broadway,29987,23674,-6313
6723.10,E 58 St & 1 Ave (NW Corner),10220,3920,-6300
5348.10,Grand St & Wythe Ave,10304,4244,-6060
4479.10,Schermerhorn St & Hoyt St,14146,8683,-5463
6740.10,W 51 St & 6 Ave,9519,4122,-5397
5997.10,W 12 St & Hudson St,12930,7560,-5370
5847.10,Greenwich St & Christopher St,9925,4722,-5203
5669.10,St Marks Pl & 2 Ave,14494,9353,-5141
7230.10,W 74 St & Columbus Ave,7604,2944,-4660


In [0]:
# Register the cleaned DataFrame as a SQL table name
trips_clean.createOrReplaceTempView("trips")

# SQL Query 1: ride length by bike type and rider type
sql_bike_type = spark.sql("""
    SELECT
        rideable_type,
        member_casual,
        COUNT(*)                                       AS num_rides,
        ROUND(AVG(duration_min), 2)                    AS avg_duration_min,
        ROUND(PERCENTILE_APPROX(duration_min, 0.5), 2) AS median_duration_min
    FROM trips
    GROUP BY rideable_type, member_casual
    ORDER BY rideable_type, member_casual
""")
display(sql_bike_type)

rideable_type,member_casual,num_rides,avg_duration_min,median_duration_min
classic_bike,casual,507993,23.4,16.27
classic_bike,member,2224547,11.84,8.03
electric_bike,casual,1629292,17.89,12.48
electric_bike,member,5841776,11.97,9.15


In [0]:
# SQL Query 2: average rides per day, weekday vs weekend, by rider type
# dayofweek(): 1 = Sunday, 7 = Saturday
sql_weekday = spark.sql("""
    SELECT
        CASE WHEN dayofweek(ride_date) IN (1, 7) THEN 'weekend' ELSE 'weekday' END AS day_type,
        member_casual,
        COUNT(*)                                  AS total_rides,
        COUNT(DISTINCT ride_date)                 AS num_days,
        ROUND(COUNT(*) / COUNT(DISTINCT ride_date)) AS avg_rides_per_day
    FROM trips
    GROUP BY 1, 2
    ORDER BY 1, 2
""")
display(sql_weekday)

day_type,member_casual,total_rides,num_days,avg_rides_per_day
weekday,casual,1302531,44,29603.0
weekday,member,5950472,44,135238.0
weekend,casual,834754,18,46375.0
weekend,member,2115851,18,117547.0


In [0]:
# Create an output volume (does nothing if it already exists)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.citibike_output")
OUTPUT = "/Volumes/workspace/default/citibike_output"

# Delete any previous output first, so stale partitions (e.g. 2026-06-30) cannot linger
dbutils.fs.rm(f"{OUTPUT}/trips_clean", recurse=True)
# Write the cleaned trips as Parquet, one folder per day.
# repartition by the same column first so each day becomes one file instead of many tiny files.
(
    trips_clean
    .repartition("ride_date")
    .write
    .mode("overwrite")
    .partitionBy("ride_date")
    .parquet(f"{OUTPUT}/trips_clean")
)

# The station table is tiny, so write it as a single file
station_flow.coalesce(1).write.mode("overwrite").parquet(f"{OUTPUT}/station_flow")

print("Write complete")

Write complete


In [0]:
# Verify the output: number of date partitions and the date range actually stored
partitions = [f.name for f in dbutils.fs.ls(f"{OUTPUT}/trips_clean") if f.name.startswith("ride_date=")]
print(f"Partitions written: {len(partitions)}  (first: {partitions[0]}, last: {partitions[-1]})")

check = spark.read.parquet(f"{OUTPUT}/trips_clean")
check.agg(
    F.min("ride_date").alias("min_date"),
    F.max("ride_date").alias("max_date"),
    F.count("*").alias("rows"),
).show()

Partitions written: 62  (first: ride_date=2026-07-01/, last: ride_date=2026-08-31/)
+----------+----------+--------+
|  min_date|  max_date|    rows|
+----------+----------+--------+
|2026-07-01|2026-08-31|10203608|
+----------+----------+--------+



In [0]:
# Read the Parquet output back
trips_pq = spark.read.parquet(f"{OUTPUT}/trips_clean")

def casual_rides_first_week(df):
    """Same query for any source: casual rides in Aug 1-7, counted by hour."""
    return (
        df
        .filter(F.col("ride_date").between("2026-08-01", "2026-08-07"))  # partition column
        .filter(F.col("member_casual") == "casual")                       # regular column
        .select("start_hour")                                             # only 1 column needed
        .groupBy("start_hour")
        .count()
    )

# Show the physical plan for the Parquet version
casual_rides_first_week(trips_pq).explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [3]: [member_casual#14911, start_hour#14913, ride_date#14914]
DictionaryFilters: [(member_casual#14911 = casual)]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/citibike_output/trips_clean]
PartitionFilters: [isnotnull(ride_date#14914), (ride_date#14914 >= 2026-08-01), (ride_date#14914 <= 2026-08-07)]
ReadSchema: struct<member_casual:string,start_hour:int>
RequiredDataFilters: [isnotnull(member_casual#14911), (member_casual#14911 = casual)]

(2) PhotonProject
Input [3]: [member_casual#14911, start_hour#14913, ride_date#14914]
A

In [0]:
# Same query on the CSV-based DataFrame, for comparison
casual_rides_first_week(trips_clean).explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- == Initial Plan ==
   PhotonResultStage (11)
   +- PhotonColumnarToRow (10)
      +- PhotonGroupingAgg (9)
         +- PhotonShuffleExchangeSource (8)
            +- PhotonShuffleMapStage (7)
               +- PhotonShuffleExchangeSink (6)
                  +- PhotonGroupingAgg (5)
                     +- PhotonProject (4)
                        +- PhotonFilter (3)
                           +- PhotonRowToColumnar (2)
                              +- Scan csv  (1)


(1) Scan csv 
Output [5]: [started_at#15471, ended_at#15472, start_station_id#15474, end_station_id#15476, member_casual#15481]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/citibike/202607-citibike-tripdata/202607-citibike-tripdata_1.csv, ... 10 entries]
PushedFilters: [IsNotNull(started_at), IsNotNull(member_casual), IsNotNull(start_station_id), IsNotNull(end_station_id), EqualTo(member_casual,casual)]
ReadSchema: struct<started_at:string,ended_a

In [0]:
import time

def time_query(df, label):
    """Run the query (an action) and report wall-clock time."""
    start = time.time()
    casual_rides_first_week(df).collect()
    elapsed = time.time() - start
    print(f"{label:<10} {elapsed:6.2f} s")
    return elapsed

csv_time = time_query(trips_clean, "CSV")
pq_time = time_query(trips_pq, "Parquet")
print(f"Speedup: {csv_time / pq_time:.1f}x")

CSV          3.31 s
Parquet      0.57 s
Speedup: 5.8x


In [0]:
import time

# Step 1: chain several transformations (lazy: Spark only records a plan)
t0 = time.time()
lazy_df = (
    trips_pq
    .filter(F.col("rideable_type") == "electric_bike")
    .withColumn("is_long_ride", F.col("duration_min") > 30)
    .groupBy("is_long_ride")
    .count()
)
t1 = time.time()
print(f"Transformations defined in {t1 - t0:.3f} s  (no data read yet)")

# Step 2: call an action (eager: Spark now reads data and runs the plan)
result = lazy_df.collect()
t2 = time.time()
print(f"Action collect() finished in {t2 - t1:.3f} s")
print(result)

Transformations defined in 0.001 s  (no data read yet)
Action collect() finished in 0.876 s
[Row(is_long_ride=True, count=520300), Row(is_long_ride=False, count=6950768)]


In [0]:
# Bonus: try caching. Serverless compute may not support cache(), so handle that gracefully.
try:
    cached_df = trips_pq.filter(F.col("member_casual") == "casual").cache()

    t0 = time.time(); cached_df.count(); t1 = time.time()   # 1st run: reads data and fills cache
    cached_df.count(); t2 = time.time()                     # 2nd run: should read from memory

    print(f"1st count (fills cache):  {t1 - t0:.2f} s")
    print(f"2nd count (from cache):   {t2 - t1:.2f} s")
    cached_df.unpersist()
except Exception as e:
    print("cache() is not available on this compute:")
    print(type(e).__name__, str(e)[:300])

1st count (fills cache):  0.56 s
2nd count (from cache):   0.46 s
